In [1]:
# there no change in the first several cells from last lecture

In [9]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

from nnzh.data import VOCAB_SIZE, build_dataset, build_vocab, load_words, split_words

words = load_words()
stoi, itos = build_vocab(words)
tr_words, va_words, te_words = split_words(words)      # test 全程不碰
print(len(words), VOCAB_SIZE)

32033 27


In [10]:
block_size = 3
Xtr,  Ytr  = build_dataset(tr_words, stoi, block_size)
Xdev, Ydev = build_dataset(va_words, stoi, block_size)
Xte,  Yte  = build_dataset(te_words, stoi, block_size)
print(Xtr.shape, Xdev.shape, Xte.shape)

torch.Size([182819, 3]) torch.Size([22760, 3]) torch.Size([22567, 3])


In [11]:
# utility function we will use later when comparing manual gradients to PyTorch gradients
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()          # 逐位相同
    app = torch.allclose(dt, t.grad)             # 数学正确就够了，看这一列
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}')

In [14]:
n_embd = 10    # the dimensionality of the character embedding vectors
n_hidden = 64  # the number of neurons in the hidden layer of the MLP

g = torch.Generator().manual_seed(2147483647)  # for reproducibility
C  = torch.randn((VOCAB_SIZE, n_embd),            generator=g)
# Layer 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1   # 只为让梯度非零，正常应为 0
# Layer 2
W2 = torch.randn((n_hidden, VOCAB_SIZE),          generator=g) * 0.1
b2 = torch.randn(VOCAB_SIZE,                      generator=g) * 0.1
# BatchNorm parameters
bngain = torch.randn((1, n_hidden)) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden)) * 0.1

# 注意：这里刻意把很多参数初始化成小随机数而不是 0。
# 全 0 会让某些梯度恰好也是 0，掩盖掉写错的公式——测不出来的 bug 最贵。
parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

4137


In [15]:
batch_size = 32
n = batch_size                      # 后面公式里到处要用，单独起个短名字
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]           # batch X, Y

In [16]:
# forward pass, "chunkated" into smaller steps that are possible to backward one at a time
emb = C[Xb]                              # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1)      # concatenate the vectors
# Linear layer 1
hprebn = embcat @ W1 + b1                # hidden layer pre-activation
# BatchNorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n-1)*(bndiff2).sum(0, keepdim=True)   # Bessel 校正：无偏方差用 n-1
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
# Non-linearity
h = torch.tanh(hpreact)                  # hidden layer
# Linear layer 2
logits = h @ W2 + b2                     # output layer
# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes       # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdims=True)
counts_sum_inv = counts_sum**-1          # 写成 **-1 而不是 1/x，方便单独反传这一步
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# PyTorch backward pass
for p in parameters:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv,   # afaik there is no cleaner way
          norm_logits, logit_maxes, logits, h, hpreact, bnraw,
          bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani,
          embcat, emb]:
    t.retain_grad()
loss.backward()
loss

tensor(3.4033, grad_fn=<NegBackward0>)